# Real A/B Test: Which Listing Presentation Feels Like Better Value?

This is a **prospective randomized survey experiment**, not a simulation. Each consenting participant sees one version of the same fictional London short-stay listing. Their perceived-value rating is compared between versions. The experiment measures how the wording/layout affects survey respondents; it does not measure actual Airbnb guest ratings or prove that any real listing will earn better reviews.

The existing Airbnb data motivates the question, but it is observational. The A/B effect can only be estimated after people are randomly assigned, see the assigned version, and submit real survey responses.


## 1. Simple study plan

- **Participants:** aim for 100 consenting adults (18+), such as course peers. Use a convenience sample and describe it that way.
- **Randomization unit:** one person. Each person sees only one version.
- **Control (A):** listing details in a plain paragraph.
- **Treatment (B):** the exact same facts, with the total price and included features presented as a clear value summary.
- **Primary outcome:** perceived value rating from 1 (very poor value) to 5 (very good value).
- **Secondary outcomes:** booking consideration (1–5) and correct recall of the all-in nightly price.
- **Assignment:** the first code cell creates a balanced, randomized list of anonymous participant codes. Give each participant the next unused code and show the version listed for that code. Keep the schedule private; do not regenerate it after recruitment begins.
- **Analysis:** compare A and B averages, report a 95% confidence interval and Welch test, and use a Mann–Whitney test as an ordinal-scale sensitivity check. Report response counts and price recall by arm.

The survey should take about two minutes. Obtain consent before showing either version. Do not collect names, email addresses, or other unnecessary personal data.


In [1]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, norm, t, ttest_ind
from IPython.display import display

_candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next((p for p in _candidates if (p / "scripts" / "ab_test_design.ipynb").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project containing scripts/ab_test_design.ipynb")
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)
SCHEDULE_PATH = DATA_DIR / "ab_test_randomization_schedule.csv"
RESPONSES_PATH = DATA_DIR / "ab_test_survey_responses.csv"
SYNTHETIC_SCHEDULE_PATH = DATA_DIR / "ab_test_randomization_schedule_synthetic.csv"
SYNTHETIC_RESPONSES_PATH = DATA_DIR / "ab_test_survey_responses_synthetic.csv"
PLANNED_N = 100
RANDOM_SEED = 2026


## 2. Generate the assignment schedule

Run this once before collecting responses. It makes 100 anonymous codes in balanced shuffled blocks, giving 50 assignments to A and 50 to B. If the schedule already exists, the notebook reuses it instead of silently replacing assignments. Keep `data/ab_test_randomization_schedule.csv` private and unchanged while the study is underway.


In [2]:
def make_assignment_schedule(n: int = PLANNED_N, seed: int = RANDOM_SEED) -> pd.DataFrame:
    if n < 4:
        raise ValueError("Plan for at least 4 participants")
    rng = np.random.default_rng(seed)
    conditions=[]
    for start in range(0, n, 4):
        size=min(4, n-start)
        block=["A_control"]*(size//2) + ["B_treatment"]*(size//2)
        if size % 2:
            block.append(rng.choice(["A_control", "B_treatment"]))
        rng.shuffle(block)
        conditions.extend(block)
    return pd.DataFrame({
        "participant_code":[f"P{i:03d}" for i in range(1,n+1)],
        "condition":conditions,
    })

if SCHEDULE_PATH.exists():
    schedule=pd.read_csv(SCHEDULE_PATH,dtype={"participant_code":"string"})
    print("Reusing the existing assignment schedule; do not regenerate during recruitment.")
else:
    schedule=make_assignment_schedule()
    schedule.to_csv(SCHEDULE_PATH,index=False)
    print("Created the private assignment schedule for use during recruitment.")
print(f"Planned participants: {len(schedule)}")
print("Assignments by condition:")
print(schedule["condition"].value_counts().sort_index().to_string())


Reusing the existing assignment schedule; do not regenerate during recruitment.
Planned participants: 100
Assignments by condition:
condition
A_control      50
B_treatment    50


## 3. Show these two listing versions

Use the text exactly as written. Keep the layout, font, and any accompanying image the same between versions; change only the order and emphasis of the same facts.

### A — Control: plain presentation

> One-bedroom apartment in Stratford for two guests. Wi-Fi, kitchen, and washing machine. Stratford station is a 15-minute walk away. £120 total per night, including cleaning and service fees.

### B — Treatment: clear value summary

> £120 total per night, including cleaning and service fees. Your stay includes the entire one-bedroom apartment for two guests, Wi-Fi, kitchen, and washing machine. Stratford station is a 15-minute walk away.

Do not show both versions to the same person. Give each participant the version matching their code in the assignment schedule.

### Ask every participant the same questions

1. **Perceived value (primary):** “Based on the listing, how good does this stay seem for the price?” 1 = Very poor value; 2 = Poor value; 3 = Neither; 4 = Good value; 5 = Very good value.
2. **Booking consideration (secondary):** “How likely would you be to consider booking this stay?” 1 = Very unlikely to 5 = Very likely.
3. **Price recall (check):** “What is the total nightly price including fees?” Correct answer: £120.



## 4. Sample-size context

With a convenience sample of 100 (50 per arm), the study is best described as a class-project experiment or pilot. For context only, if the outcome standard deviation is about 1 point on the 1–5 scale, 50 participants per arm gives roughly 80% power only for a fairly large difference of about 0.56 points at two-sided 5% significance. This is an approximation, not a guarantee; small effects may be missed. Recruit more participants if feasible and report the confidence interval regardless of significance.


In [3]:
# Approximate detectable difference with the planned equal sample size.
ASSUMED_SD = 1.0  # planning assumption on the 1-5 response scale
ALPHA = 0.05
TARGET_POWER = 0.80
n_each = int((schedule["condition"].value_counts().min()))
minimum_detectable_difference = (norm.ppf(1-ALPHA/2)+norm.ppf(TARGET_POWER))*ASSUMED_SD*math.sqrt(2/n_each)
print(f"Planned: {n_each} participants per arm")
print(f"Approximate detectable difference at 80% power: {minimum_detectable_difference:.2f} points")
print("Assumed SD=1.0; planning approximation only.")


Planned: 50 participants per arm
Approximate detectable difference at 80% power: 0.56 points
Assumed SD=1.0; planning approximation only.


## 5. Enter real survey responses

Export or enter responses into `data/ab_test_survey_responses.csv` with one row per completed participant and exactly these columns. The notebook joins each participant code to its randomized condition; do not guess or reassign conditions after seeing answers.

| Column | Values |
| --- | --- |
| `participant_code` | Code from the private randomization schedule, e.g. `P001` |
| `perceived_value` | Integer 1–5; primary outcome |
| `booking_intent` | Integer 1–5; secondary outcome |
| `price_check_correct` | `True`/`False` (or `Yes`/`No`) |

Keep the file local. Do not add names, email addresses, or free-text personal details. The notebook keeps synthetic responses in a separate file and labels their results clearly. Real responses, if present, take precedence and are matched to the real schedule.


## Synthetic dataset for a complete analysis walkthrough

This cell generates 100 **synthetic** respondents on a separate randomized schedule. The assumed treatment effect and noise are declared in the code. The synthetic file is kept separate from real survey responses; the analysis cell uses actual responses if they exist, otherwise it runs the synthetic demonstration. Synthetic findings are not evidence about real people or Airbnb guests.


In [4]:
SIM_SEED = 2027
SIM_SD = 1.0
SIM_VALUE_EFFECT = 0.50   # assumed shift on latent value rating, inserted for demonstration
SIM_BOOKING_EFFECT = 0.25 # assumed shift on latent booking-interest rating
SIM_PRICE_RECALL_A = 0.70
SIM_PRICE_RECALL_B = 0.85

if SYNTHETIC_SCHEDULE_PATH.exists() and SYNTHETIC_RESPONSES_PATH.exists():
    synthetic_schedule = pd.read_csv(SYNTHETIC_SCHEDULE_PATH, dtype={"participant_code":"string"})
    synthetic_responses = pd.read_csv(SYNTHETIC_RESPONSES_PATH, dtype={"participant_code":"string"})
    print("Reusing existing synthetic demo files.")
else:
    synthetic_schedule = make_assignment_schedule(PLANNED_N, seed=SIM_SEED)
    synthetic_schedule["participant_code"] = [f"SIM{i:03d}" for i in range(1, len(synthetic_schedule)+1)]
    sim_rng = np.random.default_rng(SIM_SEED)
    is_b = synthetic_schedule["condition"].eq("B_treatment").to_numpy()
    latent_value = sim_rng.normal(3.0 + SIM_VALUE_EFFECT*is_b, SIM_SD)
    latent_booking = sim_rng.normal(3.0 + SIM_BOOKING_EFFECT*is_b, SIM_SD)
    perceived_value = np.clip(np.rint(latent_value), 1, 5).astype(int)
    booking_intent = np.clip(np.rint(latent_booking), 1, 5).astype(int)
    price_probability = np.where(is_b, SIM_PRICE_RECALL_B, SIM_PRICE_RECALL_A)
    price_check_correct = sim_rng.random(len(is_b)) < price_probability
    synthetic_responses = pd.DataFrame({
        "participant_code": synthetic_schedule["participant_code"],
        "perceived_value": perceived_value,
        "booking_intent": booking_intent,
        "price_check_correct": price_check_correct,
    })
    synthetic_schedule.to_csv(SYNTHETIC_SCHEDULE_PATH, index=False)
    synthetic_responses.to_csv(SYNTHETIC_RESPONSES_PATH, index=False)
    print("Created separate synthetic demo data; actual response/schedule files were not overwritten.")
print(f"Synthetic respondents: {len(synthetic_responses)}")
print("Synthetic randomized assignment counts:")
print(synthetic_schedule["condition"].value_counts().sort_index().to_string())


Reusing existing synthetic demo files.
Synthetic respondents: 100
Synthetic randomized assignment counts:
condition
A_control      50
B_treatment    50


In [5]:
def welch_difference(a: pd.Series, b: pd.Series) -> dict:
    """Difference is B minus A; returns mean difference, Welch 95% CI, p-value, and group means."""
    a=pd.to_numeric(a,errors="coerce").dropna().to_numpy(dtype=float)
    b=pd.to_numeric(b,errors="coerce").dropna().to_numpy(dtype=float)
    if len(a)<2 or len(b)<2:
        raise ValueError("At least two valid responses per condition are needed for the comparison")
    estimate=float(b.mean()-a.mean())
    va,vb=a.var(ddof=1)/len(a),b.var(ddof=1)/len(b)
    se=math.sqrt(va+vb)
    df=(va+vb)**2/(va**2/(len(a)-1)+vb**2/(len(b)-1)) if se else np.inf
    critical=t.ppf(0.975,df)
    test=ttest_ind(b,a,equal_var=False)
    return {"A_n":len(a),"A_mean":float(a.mean()),"B_n":len(b),"B_mean":float(b.mean()),
            "B_minus_A":estimate,"CI95_low":estimate-critical*se,"CI95_high":estimate+critical*se,
            "Welch_p_value":float(test.pvalue),
            "Mann_Whitney_p_value":float(mannwhitneyu(b,a,alternative="two-sided").pvalue)}

if RESPONSES_PATH.exists():
    responses = pd.read_csv(RESPONSES_PATH, dtype={"participant_code":"string"})
    analysis_schedule = schedule
    print("Analyzing actual collected survey responses.")
elif SYNTHETIC_RESPONSES_PATH.exists():
    responses = pd.read_csv(SYNTHETIC_RESPONSES_PATH, dtype={"participant_code":"string"})
    analysis_schedule = synthetic_schedule
    print("SYNTHETIC DEMONSTRATION ONLY: these are generated responses with an assumed effect, not real participant data.")
else:
    responses = None
    print("No responses or synthetic demo file found; no test result can be calculated.")

if responses is not None:
    required={"participant_code","perceived_value","booking_intent","price_check_correct"}
    missing=required.difference(responses.columns)
    if missing:
        raise ValueError(f"Response file missing columns: {sorted(missing)}")
    if responses["participant_code"].isna().any() or responses["participant_code"].duplicated().any():
        raise ValueError("Each response must have one unique participant_code")
    data=responses.merge(analysis_schedule,on="participant_code",how="left",validate="one_to_one",indicator=True)
    if (data["_merge"]!="both").any():
        raise ValueError("Some response codes are not in the selected assignment schedule")
    data=data.drop(columns="_merge")
    for col in ["perceived_value","booking_intent"]:
        data[col]=pd.to_numeric(data[col],errors="coerce")
        if not data[col].dropna().between(1,5).all():
            raise ValueError(f"{col} values must be from 1 to 5")
    counts=data.groupby("condition").size().reindex(["A_control","B_treatment"],fill_value=0)
    assigned=analysis_schedule["condition"].value_counts().reindex(["A_control","B_treatment"],fill_value=0)
    print("Response counts and rates by randomized condition:")
    display(pd.DataFrame({"responses":counts,"assigned":assigned,"response_rate":counts/assigned}))

    for outcome in ["perceived_value","booking_intent"]:
        a=data.loc[data["condition"]=="A_control",outcome]
        b=data.loc[data["condition"]=="B_treatment",outcome]
        print(f"\n{outcome}: B treatment minus A control")
        display(pd.DataFrame([welch_difference(a,b)]))

    data["price_check_correct"]=data["price_check_correct"].astype("string").str.strip().str.lower().isin({"true","1","yes","correct"})
    print("Price recall accuracy by randomized condition:")
    display(data.groupby("condition")["price_check_correct"].agg(["mean","count"]).rename(columns={"mean":"share_correct","count":"n"}))


SYNTHETIC DEMONSTRATION ONLY: these are generated responses with an assumed effect, not real participant data.
Response counts and rates by randomized condition:

perceived_value: B treatment minus A control

booking_intent: B treatment minus A control
Price recall accuracy by randomized condition:


 responses  assigned  response_rate
        50        50            1.0
        50        50            1.0

 A_n  A_mean  B_n  B_mean  B_minus_A  CI95_low  CI95_high  Welch_p_value  Mann_Whitney_p_value
  50    2.82   50    3.34       0.52  0.106058   0.933942       0.014346              0.014147

 A_n  A_mean  B_n  B_mean  B_minus_A  CI95_low  CI95_high  Welch_p_value  Mann_Whitney_p_value
  50     3.0   50    3.56       0.56  0.174703   0.945297       0.004831              0.010385

 share_correct  n
          0.70 50
          0.78 50

## 6. How to interpret the real result

- A positive `B_minus_A` for perceived value means the clear value summary received higher average ratings among participants who responded.
- Use the confidence interval as the main uncertainty summary. A p-value above 0.05 does not prove the versions are equivalent, especially with a small sample.
- The Mann–Whitney p-value is a sensitivity check because a 1–5 rating is ordinal. Do not treat either test as a measure of actual booking behavior.
- Compare response rates by randomized condition; different response rates may bias the observed comparison.
- Treat the convenience sample as an exploratory result. The test supports a causal statement about the **wording shown to these respondents**, not about actual stays, actual guest reviews, or all Airbnb customers.
- If the value-focused wording improves perceived value without reducing price comprehension, the business insight is that clear, prominent explanation of inclusions may improve prospective guests’ value perception. It does not show that a host should raise prices or that a real listing’s service quality improved.

## Ethics and privacy

Use informed consent, recruit adults, avoid collecting direct identifiers, and tell participants they may stop at any time. Use a fictional listing card; do not expose a real host or listing. Keep the assignment schedule and response file private. Report only aggregate condition-level results.
